# DAI Mission — Proposal
**Data & AI in Economics | TU Dortmund**


## 1. Team

| Role | Name | 
|------|------|
| Lead | Mariia Hrechyn| 
| Member | Lutz Weiland | 


## 2. Mission Title & Research Question

**Title:** *From Balance Sheets to Buy Signals: Causal, Supervised, and Unsupervised Analysis of Fundamental Stock Return Prediction*

**Research question:**  
*Can fundamental financial ratios derived from income statements, balance sheets, and cash-flow statements reliably predict 1-year stock return direction (buy / hold / sell) for US-listed firms? 
Specifically: 

- (i) which latent fundamental profiles (clusters) carry the most predictive information, 
- (ii) does profitability (ROE) causally drive future return class after controlling for sector, leverage, size, book-to-market, and investment growth, and 
- (iii) do ensemble classifiers outperform a **Fama-French 5-factor (FF5)** economic baseline because they learn stronger class-specific signals from accounting variables, or because they better resolve uncertainty in the ambiguous hold class?*

**Why it matters:**  
*Fundamental analysis is central to active investment management, yet it is rarely subjected to rigorous causal scrutiny — most ML studies on financial data report predictive correlations without separating genuine economic drivers from sector-level confounders. This project applies three complementary lenses to a self-constructed dataset of US public-company fundamentals (SimFin, CC BY-NC 4.0): unsupervised clustering to discover latent firm archetypes, causal inference to distinguish structural profitability drivers from correlates, and supervised classification benchmarked against the Fama-French 5-factor model. Requiring all three lenses to inform each other tests whether data-driven fundamental analysis adds genuine economic value beyond established factor heuristics, or merely re-packages known market risks in a more complex form.*

## 3. Data

**Primary Source:** SimFin — Fundamental Financial Data for US-listed Companies  
Provider: SimFin ApS (Denmark)  
Access: https://simfin.com/data/access/api (free tier, Python package `simfin`)  
Licence: **CC BY-NC 4.0** — free for academic use with attribution.

---

### 3a. Raw Dataset Retrieved (`simfin_data_retrieval.ipynb`)

The data retrieval script (`simfin_data_retrieval.ipynb`) fetches all available annual US fundamentals from the SimFin API and merges them into a single panel. The raw panel saved as `simfin_raw_panel.csv` has the following characteristics:

| Property | Value |
|----------:|-------|
| Rows (company × fiscal-year) | **18,088** |
| Columns (raw features + metadata) | **86+** |
| Unique tickers | **4,787** |
| Fiscal years covered | **2020 – 2024** |
| Currency filter | USD only |
| Statement types merged | Income statement (standard / bank / insurance), Balance sheet, Cash-flow statement |

---

### 3b. Target Variable Construction

**Step 1 — 1-year forward total return (`Perform`)**  

```
Perform = (Price_{t+252} − Price_{t}) / Price_{t}
```

Observations where either price is unavailable are left as `NaN` and excluded from modelling. This yields **13,601 non-null return observations** out of 18,088 total.

**Step 2 — Ternary class label (`Class`) —  Calendar-Quarter Cohort Labeling**


| Class | Label | Meaning |
|------:|-------|--------|
| **−1** | Sell | Bottom tercile within calendar-quarter cohort |
| **0** | Hold | Middle tercile |
| **+1** | Buy | Top tercile |

---

### 3c. Pre-processing Plan (`data_preprocessing.ipynb`)


**Temporal split:**

| Partition | Fiscal Years | Purpose |
|-----------|-------------|--------|
| Training | FY 2020–2022 | Model fitting, imputation statistics |
| Validation | FY 2023 | Hyperparameter tuning |
| Test (holdout) | FY 2024 | Final evaluation |

1. **Missing value inspection** — Per-column missing rates; columns > 50% missing dropped.
2. **Sector-median imputation** — Fit on training only; frozen and applied to all splits.
3. **Winsorisation** — 1st–99th percentile bounds from training only.
4. **Z-score standardisation** — Mean and std from training only.

---

**Derived financial ratios** (winsorised at 1st–99th pct; training-set bounds):

| Ratio | Formula | Role |
|-------|---------|------|
| `ROE` | `Net Income (Common)` / `Total Equity` | **Causal treatment** — equity profitability |
| `ROA` | `Net Income` / `Total Assets` | Feature — asset profitability |
| `Leverage` | `Total Liabilities` / `Total Equity` | **Confounder** — financial risk |
| `GrossMargin` | `Gross Profit` / `Revenue` | Feature — pricing power |
| `OperatingMargin` | `Operating Income (Loss)` / `Revenue` | Feature — operational efficiency |
| `CurrentRatio` | `Total Current Assets` / `Total Current Liabilities` | Feature — short-term liquidity |
| `DebtRatio` | `Total Liabilities` / `Total Assets` | Feature — solvency |
| `CFO_Margin` | `Net Cash from Operating Activities` / `Revenue` | Feature — cash generation quality |
| `Size` | `log(MarketCap)` | **Confounder (SMB proxy)** — firm size |
| `BookToMarket` | `Total Equity / MarketCap` | **Confounder (HML proxy)** — value factor |
| `InvestmentGrowth` | YoY % change in `Total Assets` | **Confounder (CMA proxy)** — investment factor |

---

**Potential data quality issues:**


- **Missing values:** Handled via training-set sector-median imputation; columns > 50% missing dropped.
- **Extreme outliers:** Winsorisation at 1st–99th percentile (training-set bounds only).
- **Macro anomalies:** COVID-19 (2020) and rate-hiking cycle (2022) are anomalous macro periods. Controlled via NBER recession indicator feature and year-fixed effects in causal models.

---

**Income Statement columns** (`fin_type`)

| Column | Type | Role |
|--------|------|------|
| `Revenue` | Continuous | Feature / ratio denominator |
| `Cost of Revenue` | Continuous | Feature — direct production costs |
| `Gross Profit` | Continuous | Feature / GrossMargin numerator |
| `Operating Expenses` | Continuous | Feature — overhead burden |
| `Selling, General & Administrative` | Continuous | Feature — overhead spending |
| `Operating Income (Loss)` | Continuous | Feature / OperatingMargin numerator |
| `Non-Operating Income (Loss)` | Continuous | Feature — one-off gains/losses |
| `Interest Expense, Net` | Continuous | Feature — cost of debt financing |
| `Pretax Income (Loss), Adj.` | Continuous | Feature — adjusted earnings before tax |
| `Abnormal Gains (Losses)` | Continuous | Feature — earnings quality signal |
| `Pretax Income (Loss)` | Continuous | Feature — earnings before tax |
| `Income Tax (Expense) Benefit, Net` | Continuous | Feature — tax charge |
| `Income (Loss) from Continuing Operations` | Continuous | Feature — profit from ongoing business |
| `Net Income` | Continuous | Feature / ROA numerator |
| `Net Income (Common)` | Continuous | Feature / **ROE numerator** |

**Balance Sheet columns** (`fin_type_bal`)

| Column | Type | Role |
|--------|------|------|
| `Cash, Cash Equivalents & Short Term Investments` | Continuous | Feature — liquidity buffer |
| `Accounts & Notes Receivable` | Continuous | Feature — uncollected revenue |
| `Inventories` | Continuous | Feature — goods held for sale |
| `Total Current Assets` | Continuous | Feature / CurrentRatio numerator |
| `Property, Plant & Equipment, Net` | Continuous | Feature — long-term physical assets |
| `Other Long Term Assets` | Continuous | Feature — intangibles, deferred taxes |
| `Total Noncurrent Assets` | Continuous | Feature — total long-term asset base |
| `Total Assets` | Continuous | Feature / ROA & DebtRatio denominator |
| `Payables & Accruals` | Continuous | Feature — obligations to suppliers |
| `Short Term Debt` | Continuous | Feature — debt maturing within one year |
| `Total Current Liabilities` | Continuous | Feature / CurrentRatio denominator |
| `Long Term Debt` | Continuous | **Confounder** — leverage proxy in the causal DAG |
| `Total Noncurrent Liabilities` | Continuous | Feature — long-term obligations |
| `Total Liabilities` | Continuous | Feature / Leverage & DebtRatio numerator |
| `Share Capital & Additional Paid-In Capital` | Continuous | Feature — equity raised from shareholders |
| `Retained Earnings` | Continuous | Feature — accumulated past profits |
| `Total Equity` | Continuous | Feature / **ROE & Leverage denominator** |
| `Total Liabilities & Equity` | Continuous | Feature — balance check |

**Cash-Flow Statement columns** (`fin_type_cf`)

| Column | Type | Role |
|--------|------|------|
| `Net Income/Starting Line` | Continuous | Feature — accrual profit starting point |
| `Depreciation & Amortization_cf` | Continuous | Feature — non-cash add-back |
| `Non-Cash Items` | Continuous | Feature — stock-based compensation |
| `Change in Working Capital` | Continuous | Feature — accruals quality signal |
| `Net Cash from Operating Activities` | Continuous | Feature / **CFO_Margin numerator** |
| `Change in Fixed Assets & Intangibles` | Continuous | Feature — capex proxy |
| `Net Cash from Investing Activities` | Continuous | Feature — net capital deployed |
| `Cash from (Repayment of) Debt` | Continuous | Feature — debt financing activity |
| `Cash from (Repurchase of) Equity` | Continuous | Feature — share buybacks/issuance |
| `Net Cash from Financing Activities` | Continuous | Feature — net capital raised |
| `Net Change in Cash` | Continuous | Feature — end-to-end liquidity change |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Load cleaned & imputed dataset ──────────────────────────────────────────
df_raw = pd.read_csv('simfin_cleaned_imputed.csv')

# Keep only rows where Class and Perform are available
df = df_raw[df_raw['Class'].notna() & df_raw['Perform'].notna()].copy()
df['Class'] = df['Class'].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Unique tickers: {df['Ticker'].nunique()}")
print(f"Fiscal years: {sorted(df['FiscalYear'].unique())}")
df.head(3)

In [ ]:
# ── Derive financial ratios ──────────────────────────────────────────────────
def safe_div(a, b):
    """Division robust to zero denominators."""
    return a / b.replace(0, np.nan)

df['ROE']             = safe_div(df['Net Income (Common)'],              df['Total Equity'])
df['ROA']             = safe_div(df['Net Income'],                       df['Total Assets'])
df['Leverage']        = safe_div(df['Total Liabilities'],                df['Total Equity'])
df['GrossMargin']     = safe_div(df['Gross Profit'],                     df['Revenue'])
df['OperatingMargin'] = safe_div(df['Operating Income (Loss)'],          df['Revenue'])
df['CurrentRatio']    = safe_div(df['Total Current Assets'],             df['Total Current Liabilities'])
df['DebtRatio']       = safe_div(df['Total Liabilities'],                df['Total Assets'])
df['CFO_Margin']      = safe_div(df['Net Cash from Operating Activities'], df['Revenue'])

# Rev 4 confounders — include if present from updated retrieval script
NEW_CONFOUNDER_COLS = [c for c in ['Size', 'BookToMarket', 'InvestmentGrowth']
                       if c in df.columns]
BASE_RATIO_COLS = ['ROE', 'ROA', 'Leverage', 'GrossMargin',
                   'OperatingMargin', 'CurrentRatio', 'DebtRatio', 'CFO_Margin']
RATIO_COLS = BASE_RATIO_COLS + NEW_CONFOUNDER_COLS
print(f"Ratio columns: {RATIO_COLS}")

# Bounds computed on TRAINING partition only (FY2020-2022).
# Using full dataset to compute percentiles leaks val/test information.
TRAIN_YEARS = [2020, 2021, 2022]
VAL_YEARS   = [2023]
TEST_YEARS  = [2024]
train_mask  = df['FiscalYear'].isin(TRAIN_YEARS)

winsor_bounds = {}
for col in RATIO_COLS:
    if col not in df.columns: continue
    lo = df.loc[train_mask, col].quantile(0.01)
    hi = df.loc[train_mask, col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)
    winsor_bounds[col] = (lo, hi)

print("Winsorised ratios using TRAINING-SET bounds (FY2020-2022):")
df[RATIO_COLS].describe().round(3)

In [ ]:
# ── Missing values overview ──────────────────────────────────────────────────
num_cols = df.select_dtypes(include='number').columns.tolist()
miss = df[num_cols].isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]

print(f"Columns with any missing values: {len(miss)}")
print("\nMissing-rate table:")
print(miss.to_frame('missing_rate').round(4))

if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['red' if v > 0.5 else 'orange' if v > 0.1 else 'steelblue' for v in miss.values]
    miss.plot(kind='bar', color=colors, ax=ax)
    ax.set_title('Missing-value rate per column (post-imputation dataset)')
    ax.set_ylabel('Fraction missing')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values remain in numeric columns.")

In [ ]:
# ── EDA: Class distribution, Sector breakdown, Ratio correlations ────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

class_counts = df['Class'].value_counts().sort_index()
class_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#95a5a6','#2ecc71'], edgecolor='white')
axes[0].set_title('Class distribution')
axes[0].set_xlabel('Class (−1 sell / 0 hold / +1 buy)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['-1 (sell)', '0 (hold)', '+1 (buy)'], rotation=0)

sector_counts = df['Sector'].value_counts().head(10)
sector_counts.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Observations by sector')
axes[1].set_xlabel('Count')

for cls, color, label in [(-1,'#e74c3c','sell'), (0,'#95a5a6','hold'), (1,'#2ecc71','buy')]:
    axes[2].hist(df.loc[df['Class']==cls, 'Perform'].clip(-2, 3), bins=50, alpha=0.5, color=color, label=label)
axes[2].set_title('1-year forward return by class')
axes[2].set_xlabel('Perform (clipped −2 to +3)')
axes[2].legend()
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 6))
corr = df[RATIO_COLS + ['Perform']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax2, linewidths=0.5)
ax2.set_title('Correlation matrix — financial ratios & forward return')
plt.tight_layout()
plt.show()

## 4. Planned Methods

The mission applies at least one technique from each of the three mandatory blocks. Ticked items below are the methods we plan to use; each choice is justified in terms of the research question and dataset properties.


### 4a. Causal Inference

- [x] **Causal graph / DAG (DoWhy)**
- [x] **Backdoor adjustment**
- [ ] Instrumental variable
- [x] **Propensity score stratification** (robustness check)

*Justification:*  
We ask whether high return-on-equity (ROE) *causally* increases the probability of a buy signal after controlling for well-established financial confounders. A DAG is the minimal formal structure required to state that assumption precisely and to identify a valid adjustment set.


**DAG structure:**

```
Size             ──→ ROE,  FwdReturn
BookToMarket     ──→ ROE,  FwdReturn
InvestmentGrowth ──→ ROE,  FwdReturn
Leverage         ──→ ROE,  FwdReturn
Sector           ──→ ROE,  FwdReturn
ROE              ──→ FwdReturn        ← path of interest (ATE)
```

**Edge rationale:**
- *Size → ROE*: larger firms face more competitive pressure on margins; size mechanically scales equity.
- *Size → FwdReturn*: Fama-French SMB factor — small firms earn a return premium.
- *BookToMarket → ROE*: high book-to-market (value) firms tend to have lower profitability.
- *BookToMarket → FwdReturn*: Fama-French HML value premium.
- *InvestmentGrowth → ROE*: rapid asset accumulation dilutes asset productivity ratios.
- *InvestmentGrowth → FwdReturn*: Fama-French CMA factor — conservative firms earn higher returns.
- *Leverage → ROE*: DuPont identity — leverage amplifies ROE mechanically.
- *Leverage → FwdReturn*: financial distress risk premium.
- *Sector → ROE, FwdReturn*: industry-level capital structure norms and return differentials.
- *ROE → FwdReturn*: our causal hypothesis.

**Identification:** backdoor adjustment on {Size, BookToMarket, InvestmentGrowth, Leverage, Sector} blocks all confounding paths into ROE and satisfies the backdoor criterion.

**Estimation methods:**
- Primary: propensity-score stratification (ROE tercile as treatment level)
- Robustness: doubly-robust AIPW estimator

**Validation:** ≥ 2 DoWhy refutation tests — `add_unobserved_common_cause` and `placebo_treatment_refuter`.

**Synthesis with other blocks:** If RF/XGBoost feature importances rank ROE, Size, and BookToMarket as top predictors, this is consistent with the causal DAG. A small causal ATE alongside high supervised feature importance signals a largely correlational relationship — discussed explicitly in Section 8.

### 4b. Supervised Learning

- [ ] Linear / Ridge / Lasso regression
- [x] **Logistic regression** (Softmax / multinomial, used in the Soft Voting Ensemble)
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] **Decision Tree / Random Forest** (primary interpretable model; feature importances)
- [ ] Neural network
- [x] **Other:** XGBoost (gradient boosting), Soft Voting Ensemble (RF + XGBoost + Softmax)

*Justification:*  
The task is ternary classification (sell / hold / buy) on structured tabular data with mixed ratio and categorical features. Ensemble tree models (RF, XGBoost) are the established best practice for this data type: they handle non-linear interactions, are robust to outlier ratios, and require no feature scaling.

**Feature set:** 8 derived financial ratios + Size, BookToMarket, InvestmentGrowth (FF5 factor proxies) + NBER recession indicator + Sector dummies + Cluster membership from Section 4c.

**Temporal split:** train FY2020–2022 (NBER recession flag absorbs COVID anomaly), validate FY2023, test FY2024. All imputation / winsorisation / standardisation statistics computed on training set only.

> **Fama-French 5-Factor (FF5) Economic Baseline:**  
> The original proposal used FF3. FF3 is outdated for profitability research because it excludes the profitability (RMW) and investment (CMA) factors — directly related to the accounting-ratio features used here. Using FF3 would risk falsely attributing factor exposure to ML "skill". We upgrade to **FF5 (Fama & French, 2015)**.

**FF5 baseline procedure:**  
Monthly FF5 factors (MKT, SMB, HML, RMW, CMA) downloaded from the Kenneth French Data Library. For each firm, a trailing 12-month regression is fit:

```
R_i,t − R_f,t = α_i + β₁·MKT + β₂·SMB + β₃·HML + β₄·RMW + β₅·CMA + ε
```

Predicted excess return converted to directional class label using the same calendar-quarter cohort tercile rule as the target.

**Long-Short portfolio alpha:**  
At each calendar-quarter rebalancing date, form Long (Buy-signal) and Short (Sell-signal) equal-weighted portfolios. Regress spread returns on FF5 factors; the **FF5 alpha** is the return unexplained by the five factors. Report annualised alpha, t-statistic, and Sharpe ratio.

**Synthesis with unsupervised block:** Cluster membership from Section 4c is a supervised feature. Ablation study measures the marginal information contribution.

### 4c. Unsupervised / Generative Learning

- [x] **K-Means clustering** (segment firms into latent fundamental archetypes)
- [x] **Hierarchical clustering** (Ward linkage; dendrogram to validate k independently of K-Means)
- [ ] Variational autoencoder
- [ ] GAN
- [x] **Other:** PCA (dimensionality reduction before clustering)

*Justification:*  
The 8+ financial ratios are strongly collinear (ROE and ROA both capture profitability; Leverage and DebtRatio both capture solvency). PCA decorrelates them and projects firms into a low-dimensional fundamental space where K-Means distance metrics are more stable and interpretable. Hierarchical clustering (Ward) cross-validates the number of natural clusters via dendrogram without requiring a pre-specified k.

**Synthesis with supervised block:** The cluster label is added as a categorical feature to the supervised classifier (Section 4b). An ablation study measures the marginal information contribution of cluster membership.

**Synthesis with causal block:** If high-ROE firms cluster into a distinct 'quality' archetype, the causal ATE of ROE on Buy should be largest within that cluster — providing a consistency check on the DAG structural assumptions.

## 5. Evaluation Strategy

**Supervised learning metrics:**  
Per-class precision, recall, and F1-score (macro-averaged) as the primary statistical metric. Directional accuracy as the primary economic metric. AUC-OVR for probability calibration. All reported on the temporal hold-out set (FY2024).

**Economic baselines:**
- *Majority-class baseline:* always predict the most frequent class.
- *Random baseline:* uniform random prediction.
- *Fama-French 5-factor (FF5) baseline:* use FF5, which explicitly captures profitability (RMW) and investment (CMA) effects directly related to the accounting ratios used as features. 
- *Long-Short portfolio alpha:* annualised FF5 alpha of the Long-Buy / Short-Sell spread, with t-statistic and Sharpe ratio. Provides an economically interpretable value-add measure beyond factor exposure.

**Causal validation:**  
≥ 2 DoWhy refutation tests: `add_unobserved_common_cause` (sensitivity to hidden confounders) and `placebo_treatment_refuter` (should yield near-zero ATE).

**Unsupervised evaluation:**  
Silhouette score across k = 2–8; elbow criterion on inertia; economic profiling of clusters; ablation study with/without Cluster features.

**Survivorship bias robustness:**  
The SimFin free tier covers only currently-listed US firms; bankrupt or delisted firms are absent. Two bias mechanisms affect training: (a) *feature-level* — the model learns characteristics of survivors, not ex-ante selections; (b) *return-level* — the left tail of the return distribution is truncated, inflating Buy-signal apparent performance (Banz, 1981; Kothari et al., 1995). We quantify severity by comparing SimFin universe counts per year against Compustat/CRSP benchmarks. All performance claims are explicitly qualified with this caveat.

**Synthesis criterion:**  
(1) PCA/K-Means cluster profiles → feature engineering for the supervised block; (2) supervised feature importances → consistency check on the causal DAG; (3) causal ATE vs. supervised performance → distinguishes structural drivers from correlates; (4) ML vs. FF5 + Long-Short alpha → economic value-add.

## 6. Work Plan

| # | Task | Owner | Deliverable |
|---|------|-------|-------------|
| 1 | Data retrieval — SimFin fundamentals + prices; forward returns; calendar-quarter cohort class labels; NBER recession flag; Size/BtM/InvestmentGrowth | Mariia | `simfin_data_retrieval.ipynb`, `simfin_dataset.csv` |
| 2 | Data cleaning & imputation — drop high-missing columns; training-set sector-median imputation; leakage-free winsorisation & z-score | Lutz | `data_preprocessing.ipynb`, `simfin_cleaned_imputed.csv` |
| 3 | EDA & ratio derivation — derive 8 financial ratios + 3 FF5 proxies; class/sector/return distribution plots; correlation heatmap | Mariia | Executed cells in proposal (Section 3) |
| 4 | Unsupervised block — z-score scaling, PCA (≥80% variance), k-means (elbow + silhouette), Ward hierarchical clustering, cluster profiling | Mariia | Section 7c |
| 5 | Causal inference block — binarise ROE; build DoWhy CausalModel with 5-confounder DAG; backdoor ATE; propensity-score + AIPW robustness; ≥ 2 refutation tests | Lutz | Section 7a |
| 6 | Supervised learning block — temporal train/val/test split; RF + XGBoost + ensemble; cost-sensitive rules; SHAP feature importance | Mariia | Section 7b |
| 7 | FF5 baseline + Long-Short alpha — download FF5 factors; fit factor regressions; convert to class labels; compute L-S spread alpha vs FF5 | Lutz | Section 7b (baseline comparison) |
| 8 | Survivorship bias analysis — compare SimFin vs Compustat/CRSP universe counts; document feature-level and return-level bias mechanisms | Mariia + Lutz | Section 8 |
| 9 | Synthesis & Discussion — connect cluster profiles → causal interpretation → supervised performance; limitations; conclusion | Mariia + Lutz | Section 8 |